# Spécificités Oracle : PL/SQL, DUAL, ROWNUM, Séquences

Ce document couvre les particularités syntaxiques et procédurales d'Oracle par rapport à SQL standard / SQLite, avec un jeu de données réaliste (communes normandes et relevés de pollution) repris des notebooks précédents.

La cellule de setup ci-dessous génère le script SQL complet (`04_oracle_setup.sql`) destiné à être exécuté dans un client Oracle (DataGrip ou équivalent) avant de reproduire les exemples qui suivent.

## Sommaire
1. Génération du script de création de la base
2. Configuration d'une instance Oracle XE (Docker) et connexion client
3. `DUAL` — la pseudo-table Oracle
4. `ROWNUM` vs `FETCH FIRST`
5. Séquences — l'auto-incrément à l'oracle
6. PL/SQL — blocs anonymes
7. Curseurs explicites
8. Fonctions stockées
9. Procédures stockées (paramètres IN/OUT)
10. Gestion des exceptions
11. Window functions sous Oracle
12. Récapitulatif comparatif SQLite/SQL standard ↔ Oracle
13. Connexion Python à Oracle (python-oracledb)


## 1. Génération du script de création de la base

Les données reprennent exactement celles utilisées dans les notebooks SQL et pandas avancés (communes normandes et relevés de pollution mensuels), traduites en syntaxe Oracle. Cette cellule génère le script complet en un fichier `.sql`, exécutable tel quel dans une console connectée à une base Oracle.

In [1]:
# Génération du script SQL complet (DDL + DML) en syntaxe Oracle
# Données identiques à celles des notebooks SQL et pandas avancés

communes_data = [
    (1, "Le Havre",   "Seine-Maritime", 170000, 0.9041, 21.4),
    (2, "Rouen",      "Seine-Maritime", 110000, 0.5451, 15.2),
    (3, "Dieppe",     "Seine-Maritime",  30000, 0.5500, 16.7),
    (4, "Caen",       "Calvados",       105000, 0.1771,  9.8),
    (5, "Evreux",     "Eure",            48000, 0.4200, 13.1),
    (6, "Cherbourg",  "Manche",          78000, 0.3100, 11.0),
]

import numpy as np
np.random.seed(42)

base_no2 = {1: 38.2, 2: 27.5, 3: 25.9, 4: 14.1, 5: 22.0, 6: 18.4}
mois_liste = [f"2025-{m:02d}" for m in range(1, 13)]

releves_rows = []
for commune_id, base in base_no2.items():
    for i, mois in enumerate(mois_liste):
        saison = 6 * np.cos(2 * np.pi * i / 12)
        bruit = np.random.normal(0, 1.5)
        valeur = round(base + saison + bruit, 1)
        releves_rows.append((commune_id, mois, valeur))

script_lines = []
script_lines.append("-- =========================================================")
script_lines.append("-- SQL Practice -- Script de setup Oracle")
script_lines.append("-- A executer dans un client SQL connecte a une base Oracle")
script_lines.append("-- =========================================================")
script_lines.append("")
script_lines.append("DROP TABLE releves_pollution;")
script_lines.append("DROP TABLE communes;")
script_lines.append("DROP SEQUENCE releve_seq;")
script_lines.append("")
script_lines.append("CREATE TABLE communes (")
script_lines.append("    commune_id     NUMBER PRIMARY KEY,")
script_lines.append("    nom            VARCHAR2(50) NOT NULL,")
script_lines.append("    region         VARCHAR2(50) NOT NULL,")
script_lines.append("    population     NUMBER,")
script_lines.append("    score_je       NUMBER(6,4),")
script_lines.append("    taux_pauvrete  NUMBER(5,2)")
script_lines.append(");")
script_lines.append("")

for row in communes_data:
    script_lines.append(
        f"INSERT INTO communes VALUES ({row[0]}, '{row[1]}', '{row[2]}', {row[3]}, {row[4]}, {row[5]});"
    )

script_lines.append("")
script_lines.append("CREATE SEQUENCE releve_seq START WITH 1 INCREMENT BY 1;")
script_lines.append("")
script_lines.append("CREATE TABLE releves_pollution (")
script_lines.append("    releve_id     NUMBER PRIMARY KEY,")
script_lines.append("    commune_id    NUMBER,")
script_lines.append("    mois          VARCHAR2(7),")
script_lines.append("    no2           NUMBER(5,1),")
script_lines.append("    CONSTRAINT fk_commune FOREIGN KEY (commune_id) REFERENCES communes(commune_id)")
script_lines.append(");")
script_lines.append("")

for commune_id, mois, no2 in releves_rows:
    script_lines.append(
        f"INSERT INTO releves_pollution VALUES (releve_seq.NEXTVAL, {commune_id}, '{mois}', {no2});"
    )

script_lines.append("")
script_lines.append("COMMIT;")

script = "\n".join(script_lines)

with open("04_oracle_setup.sql", "w", encoding="utf-8") as f:
    f.write(script)

print(f"Script genere : 04_oracle_setup.sql ({len(script_lines)} lignes, {len(communes_data)} communes, {len(releves_rows)} releves)")
print()
print("--- Apercu des 15 premieres lignes ---")
print("\n".join(script_lines[:15]))


Script genere : 04_oracle_setup.sql (109 lignes, 6 communes, 72 releves)

--- Apercu des 15 premieres lignes ---
-- =========================================================
-- SQL Practice -- Script de setup Oracle
-- A executer dans un client SQL connecte a une base Oracle
-- =========================================================

DROP TABLE releves_pollution;
DROP TABLE communes;
DROP SEQUENCE releve_seq;

CREATE TABLE communes (
    commune_id     NUMBER PRIMARY KEY,
    nom            VARCHAR2(50) NOT NULL,
    region         VARCHAR2(50) NOT NULL,
    population     NUMBER,
    score_je       NUMBER(6,4),


## 2. Configuration d'une instance Oracle XE et connexion client

**Option A — Docker (recommandé)**
```bash
docker run -d --name oracle-xe -p 1521:1521 -e ORACLE_PASSWORD=motdepasse gvenzl/oracle-xe:21-slim
```
L'initialisation complète de la base prend une à deux minutes après le démarrage du conteneur. Le message `DATABASE IS READY TO USE!` dans les logs (`docker logs -f oracle-xe`) confirme la disponibilité.

**Option B — Installation native**
Oracle Database XE est également disponible en téléchargement direct depuis le site Oracle — installation plus lourde, sans dépendance à Docker.

**Connexion depuis un client SQL (DataGrip ou équivalent)**
- Host : `localhost`, Port : `1521`
- Service name : `XEPDB1` (pluggable database par défaut de l'image Docker ci-dessus) — le mode de connexion doit être réglé sur *Service name*, et non *SID*, sous peine d'erreur `ORA-12505`
- Utilisateur / mot de passe : ceux définis à la création du conteneur


## 3. `DUAL` — la pseudo-table Oracle

Contrairement à SQLite ou PostgreSQL, Oracle exige toujours une clause `FROM`, même pour une simple expression sans table réelle. `DUAL` est une table système à une seule ligne et une seule colonne, prévue pour cet usage.

```sql
-- Fonctionne en SQLite/PostgreSQL, pas en Oracle :
-- SELECT 1 + 1;

-- Syntaxe Oracle :
SELECT 1 + 1 FROM DUAL;

SELECT SYSDATE FROM DUAL;              -- date/heure serveur actuelle
SELECT USER FROM DUAL;                 -- utilisateur connecté
SELECT UPPER('le havre') FROM DUAL;    -- test d'une fonction isolée
```

`DUAL` est couramment utilisée pour tester rapidement une expression ou une fonction sans dépendre d'une table existante.


## 4. `ROWNUM` vs `FETCH FIRST`

### Un piège fréquent

`ROWNUM` est attribué **avant** l'exécution du `ORDER BY`.

```sql
-- Ne retourne PAS les 3 communes au score JE le plus élevé
SELECT nom, score_je
FROM communes
WHERE ROWNUM <= 3
ORDER BY score_je DESC;
-- ROWNUM filtre sur les 3 premières lignes retournées AVANT le tri,
-- puis trie ensuite uniquement ces 3 lignes déjà sélectionnées
-- selon l'ordre physique de stockage.
```

### Correction avec sous-requête (Oracle < 12c)

```sql
SELECT * FROM (
    SELECT nom, score_je
    FROM communes
    ORDER BY score_je DESC
)
WHERE ROWNUM <= 3;
```

### Syntaxe moderne (Oracle 12c+, recommandée)

```sql
SELECT nom, score_je
FROM communes
ORDER BY score_je DESC
FETCH FIRST 3 ROWS ONLY;

-- Avec pagination (équivalent d'un OFFSET) :
SELECT nom, score_je
FROM communes
ORDER BY score_je DESC
OFFSET 3 ROWS FETCH NEXT 3 ROWS ONLY;
```

Équivalent SQLite : `... ORDER BY score_je DESC LIMIT 3` — syntaxe plus directe, introduite côté Oracle (sous la forme `FETCH FIRST`) uniquement à partir de la version 12c.


## 5. Séquences — l'auto-incrément à l'oracle

SQLite gère l'auto-incrément avec `INTEGER PRIMARY KEY AUTOINCREMENT` directement sur la colonne. Oracle (avant la version 12c) sépare cette fonctionnalité en un objet indépendant, la séquence, appelée explicitement à chaque insertion.

```sql
CREATE SEQUENCE releve_seq START WITH 1 INCREMENT BY 1;

INSERT INTO releves_pollution (releve_id, commune_id, mois, no2)
VALUES (releve_seq.NEXTVAL, 1, '2025-01', 41.3);

-- Consultation de la dernière valeur générée dans la session en cours
SELECT releve_seq.CURRVAL FROM DUAL;
```

Point d'attention : `CURRVAL` n'est utilisable que si `NEXTVAL` a déjà été appelé au moins une fois au sein de la session active — une requête `CURRVAL` isolée en tout début de session lève l'erreur `ORA-08002`.

**Alternative moderne (Oracle 12c+)** — colonne `IDENTITY`, plus proche de la syntaxe SQLite/PostgreSQL :

```sql
CREATE TABLE releves_pollution (
    releve_id  NUMBER GENERATED ALWAYS AS IDENTITY,
    commune_id NUMBER,
    mois       VARCHAR2(7),
    no2        NUMBER(5,1)
);
-- Suppression du besoin d'appeler .NEXTVAL manuellement à l'insertion
```

Le script généré en section 1 utilise volontairement la séquence classique (`releve_seq.NEXTVAL`), la forme la plus répandue sur les bases Oracle historiques en environnement d'entreprise.


## 6. PL/SQL — blocs anonymes

PL/SQL est le langage procédural propriétaire d'Oracle, qui encapsule du SQL dans une structure `DECLARE / BEGIN / END`. C'est la principale différence avec SQLite, qui reste strictement déclaratif, sans variables ni logique procédurale.

```sql
SET SERVEROUTPUT ON

DECLARE
    v_nom    communes.nom%TYPE;       -- %TYPE : hérite du type de la colonne
    v_score  communes.score_je%TYPE;
BEGIN
    SELECT nom, score_je
    INTO v_nom, v_score                -- INTO : requis pour stocker un résultat SELECT
    FROM communes
    WHERE commune_id = 1;

    DBMS_OUTPUT.PUT_LINE('Commune : ' || v_nom || ' - Score JE : ' || v_score);
END;
/
```

Éléments clés :
- `%TYPE` évite de dupliquer un type de donnée entre la table et la variable, avec synchronisation automatique en cas de modification du schéma
- `INTO` est obligatoire pour affecter le résultat d'un `SELECT` à des variables
- `DBMS_OUTPUT.PUT_LINE` produit un affichage en console, visible une fois `SERVEROUTPUT` activé (via la commande ci-dessus ou l'option correspondante du client SQL utilisé)
- Le `/` final déclenche l'exécution du bloc


## 7. Curseurs explicites

Un curseur permet de parcourir un ensemble de résultats ligne par ligne en PL/SQL, pour les cas où une opération doit être appliquée individuellement à chaque ligne.

```sql
DECLARE
    CURSOR c_communes IS
        SELECT nom, score_je
        FROM communes
        ORDER BY score_je DESC;

    v_nom    communes.nom%TYPE;
    v_score  communes.score_je%TYPE;
BEGIN
    OPEN c_communes;
    LOOP
        FETCH c_communes INTO v_nom, v_score;
        EXIT WHEN c_communes%NOTFOUND;    -- condition de sortie de boucle

        DBMS_OUTPUT.PUT_LINE(v_nom || ' : ' || v_score);
    END LOOP;
    CLOSE c_communes;   -- fermeture explicite requise
END;
/
```

Pour un simple affichage ou une agrégation, une requête SQL déclarative reste généralement préférable en termes de performance et de lisibilité. Les curseurs deviennent pertinents lorsque chaque ligne déclenche une action complexe (appel de procédure, écriture conditionnelle) plutôt qu'une simple lecture.


## 8. Fonctions stockées

Une fonction stockée encapsule une logique réutilisable directement en base, utilisable ensuite comme n'importe quelle fonction SQL native dans un `SELECT`.

```sql
CREATE OR REPLACE FUNCTION categoriser_severite (p_score IN NUMBER)
RETURN VARCHAR2
IS
BEGIN
    IF p_score >= 0.7 THEN
        RETURN 'CRITIQUE';
    ELSIF p_score >= 0.35 THEN
        RETURN 'MODÉRÉ';
    ELSE
        RETURN 'BON';
    END IF;
END;
/

-- Utilisation directe dans une requête
SELECT nom, score_je, categoriser_severite(score_je) AS severite
FROM communes
ORDER BY score_je DESC;
```

Cette fonction reproduit la logique implémentée avec `apply()` dans le notebook pandas correspondant, ainsi qu'un `CASE WHEN` équivalent en SQL standard — trois expressions différentes de la même règle métier selon la couche applicative concernée.


## 9. Procédures stockées (paramètres IN/OUT)

Contrairement à une fonction, qui retourne toujours une valeur unique via `RETURN`, une procédure peut renvoyer plusieurs résultats via des paramètres `OUT`, ou n'en renvoyer aucun.

```sql
CREATE OR REPLACE PROCEDURE stats_region (
    p_region        IN  communes.region%TYPE,
    p_moyenne       OUT NUMBER,
    p_nb_communes   OUT NUMBER
)
IS
BEGIN
    SELECT AVG(score_je), COUNT(*)
    INTO p_moyenne, p_nb_communes
    FROM communes
    WHERE region = p_region;
END;
/

-- Appel depuis un bloc anonyme
SET SERVEROUTPUT ON

DECLARE
    v_moy NUMBER;
    v_nb  NUMBER;
BEGIN
    stats_region('Seine-Maritime', v_moy, v_nb);
    DBMS_OUTPUT.PUT_LINE('Moyenne : ' || ROUND(v_moy, 3) || ' sur ' || v_nb || ' communes');
END;
/
```


## 10. Gestion des exceptions

PL/SQL propose un mécanisme structuré de gestion d'erreurs, avec des exceptions prédéfinies par Oracle (`NO_DATA_FOUND`, `TOO_MANY_ROWS`, `ZERO_DIVIDE`...) en complément des exceptions personnalisées.

```sql
SET SERVEROUTPUT ON

DECLARE
    v_score communes.score_je%TYPE;
BEGIN
    SELECT score_je INTO v_score
    FROM communes
    WHERE commune_id = 999;   -- identifiant inexistant

EXCEPTION
    WHEN NO_DATA_FOUND THEN
        DBMS_OUTPUT.PUT_LINE('Aucune commune trouvée avec cet identifiant.');
    WHEN TOO_MANY_ROWS THEN
        DBMS_OUTPUT.PUT_LINE('Plusieurs lignes trouvées, SELECT INTO en attend une seule.');
    WHEN OTHERS THEN
        DBMS_OUTPUT.PUT_LINE('Erreur inattendue : ' || SQLERRM);
END;
/
```

`SELECT ... INTO` lève automatiquement `NO_DATA_FOUND` en l'absence de ligne correspondante, et `TOO_MANY_ROWS` si plusieurs lignes sont trouvées — une variable scalaire ne pouvant recevoir qu'une seule valeur.


## 11. Window functions sous Oracle

Oracle implémente les window functions avec une syntaxe quasiment identique à SQL standard (`ROW_NUMBER()`, `RANK()`, `DENSE_RANK()`, `PARTITION BY`, `LAG()`/`LEAD()`, `ROWS BETWEEN ... PRECEDING AND CURRENT ROW`). Les requêtes du notebook SQL avancé sont directement réutilisables sans adaptation.

Seul point de vigilance : la limitation de résultats combinée à une window function doit utiliser `FETCH FIRST` (section 4) plutôt que `LIMIT`, absent d'Oracle.

```sql
SELECT
    nom,
    score_je,
    ROW_NUMBER() OVER (ORDER BY score_je DESC) AS rang
FROM communes;
```


## 12. Récapitulatif comparatif — SQLite/SQL standard ↔ Oracle

In [2]:
import pandas as pd

recap = pd.DataFrame([
    ["Sélection sans table",        "SELECT 1+1;",                          "SELECT 1+1 FROM DUAL;"],
    ["Limiter les résultats",       "LIMIT 3",                              "FETCH FIRST 3 ROWS ONLY  (ou ROWNUM avec sous-requête)"],
    ["Pagination",                  "LIMIT 3 OFFSET 3",                     "OFFSET 3 ROWS FETCH NEXT 3 ROWS ONLY"],
    ["Type texte",                  "TEXT",                                 "VARCHAR2(n)"],
    ["Type numérique",              "INTEGER / REAL",                       "NUMBER  ou  NUMBER(p,s)"],
    ["Auto-incrément",              "INTEGER PRIMARY KEY AUTOINCREMENT",    "SEQUENCE + .NEXTVAL  (ou IDENTITY en 12c+)"],
    ["Date/heure serveur",          "datetime('now')",                      "SYSDATE  (via DUAL)"],
    ["Bloc procédural",             "Non supporté nativement",              "PL/SQL : DECLARE / BEGIN / END"],
    ["Fonction personnalisée",      "Non supporté nativement",              "CREATE OR REPLACE FUNCTION ... RETURN"],
    ["Procédure stockée",           "Non supporté nativement",              "CREATE OR REPLACE PROCEDURE"],
    ["Window functions",            "Supportées (syntaxe standard)",        "Supportées (syntaxe quasi identique)"],
    ["CTE / WITH",                  "Supportées",                           "Supportées (syntaxe identique)"],
], columns=["Besoin", "SQLite / SQL standard", "Oracle"])

recap


,Besoin,SQLite / SQL standard,Oracle
0,Sélection sans table,SELECT 1+1;,SELECT 1+1 FROM DUAL;
1,Limiter les résultats,LIMIT 3,FETCH FIRST 3 ROWS ONLY (ou ROWNUM avec sous-...
2,Pagination,LIMIT 3 OFFSET 3,OFFSET 3 ROWS FETCH NEXT 3 ROWS ONLY
3,Type texte,TEXT,VARCHAR2(n)
4,Type numérique,INTEGER / REAL,"NUMBER ou NUMBER(p,s)"
5,Auto-incrément,INTEGER PRIMARY KEY AUTOINCREMENT,SEQUENCE + .NEXTVAL (ou IDENTITY en 12c+)
6,Date/heure serveur,datetime('now'),SYSDATE (via DUAL)
7,Bloc procédural,Non supporté nativement,PL/SQL : DECLARE / BEGIN / END
8,Fonction personnalisée,Non supporté nativement,CREATE OR REPLACE FUNCTION ... RETURN
9,Procédure stockée,Non supporté nativement,CREATE OR REPLACE PROCEDURE


## 13. Connexion Python à Oracle (`python-oracledb`)

Pour interroger Oracle depuis un environnement Python plutôt que depuis une console SQL dédiée — utile pour enchaîner avec des traitements pandas :

```python
# pip install oracledb

import oracledb
import pandas as pd

connection = oracledb.connect(
    user="system",
    password="motdepasse",
    dsn="localhost:1521/XEPDB1"
)

df = pd.read_sql("SELECT nom, score_je FROM communes ORDER BY score_je DESC", connection)
print(df)

connection.close()
```

Le mode « thin » de `python-oracledb` (par défaut depuis la version 1.0) ne nécessite pas l'installation du client Oracle complet, contrairement à l'ancien `cx_Oracle`.


## Synthèse

Ce document a couvert les points de divergence syntaxique et fonctionnelle entre Oracle et SQL standard/SQLite : la pseudo-table `DUAL`, la limitation de résultats (`ROWNUM` vs `FETCH FIRST`), les séquences, et l'ensemble du socle PL/SQL (blocs anonymes, curseurs, fonctions et procédures stockées, gestion des exceptions).

Les window functions et les CTEs, déjà couvertes dans le notebook SQL avancé, s'appliquent sans adaptation sur Oracle — seule la clause de limitation de résultats nécessite un ajustement de syntaxe.
